# 課題と解答例：40_network_foundations

元Notebook: [../40_network_foundations.ipynb](../40_network_foundations.ipynb)

## 課題

1. Aの第1行第2列の値と，グラフ上の対応する矢印を答える．値の符号も説明する．
2. 行列積 $A\mathbf{x}$ の第2成分を手計算し，Pythonの出力と照合する．
3. betaを0として1ステップ進め，3頂点すべてが原点へ近づくことを確認する．
4. Aの第3行第1列を-0.5から+0.5へ変える．頂点3のnetwork_inputがどのように変わるか，実行前に予想する．
5. dtを0.2から1.0へ変え，1ステップの変化が大きくなることを確認する．時間刻みはモデルの係数ではなく数値計算の設定であることを説明する．

### 追加実装課題

6. `edge_table(A)` を作り，非零要素ごとに `from`, `to`, `weight` を持つDataFrameを返す．
7. `one_step(A, x, alpha, beta, dt)` を作り，`self_decay`, `network_input`, `dxdt`, `x_next` を辞書で返す．
8. `A[2, 0]` を複数の値に変えた表を作り，頂点3の `network_input` と `x_next` がどう変わるか確認する．

## 解答例

1. `A[0, 1] = 0.8` であり，これは頂点2から頂点1への正の影響である．

2. 第2成分は

   ```{math}
   (A\mathbf{x})_2=0x_1+0x_2+0.6x_3=0.6\times0.25=0.15
   ```

   である．

3. `beta = 0` なら `dxdt = -alpha * x` なので，各成分は原点へ近づく．

4. `A[2, 0]` を `-0.5` から `+0.5` に変えると，頂点3へのネットワーク入力は `-0.3` から `+0.3` へ反転する．

5. `dt` を大きくすると，1ステップの移動量 `dt * dxdt` が大きくなる．ただし，モデル係数 `alpha`, `beta`, `A` は変わらない．

6. 非零要素表の実装例である．

   ```python
   def edge_table(A):
       rows = []
       for i in range(A.shape[0]):
           for j in range(A.shape[1]):
               if A[i, j] != 0:
                   rows.append({"from": j + 1, "to": i + 1, "weight": A[i, j]})
       return pd.DataFrame(rows)
   ```

7. 1ステップ更新の関数である．

   ```python
   def one_step(A, x, alpha, beta, dt):
       self_decay = -alpha * x
       network_input = beta * (A @ x)
       dxdt = self_decay + network_input
       x_next = x + dt * dxdt
       return {"self_decay": self_decay, "network_input": network_input, "dxdt": dxdt, "x_next": x_next}
   ```

8. `A[2, 0]` を変えた表は次のように作る．

   ```python
   rows = []
   for value in [-0.5, 0.0, 0.5]:
       A_try = A.copy()
       A_try[2, 0] = value
       result = one_step(A_try, x, alpha, beta, dt)
       rows.append({"A31": value, "network_input_3": result["network_input"][2], "x_next_3": result["x_next"][2]})
   pd.DataFrame(rows)
   ```